In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import dask.dataframe as dd
import dask
import coiled
import os
import glob
from global_snowmelt_runoff_onset.config import Config
from shapely.geometry import Point
import easysnowdata
import rasterio
import time

In [ ]:
config = Config('config/global_config_v9.txt')

In [ ]:
# cluster = coiled.Cluster(
#     name="parquets_to_aggregation_netcdfs",
#     idle_timeout="10 minutes",
#     n_workers=50,
#     worker_memory="32 GB",
#     worker_cpu=4,
#     scheduler_memory="128 GB",
#     spot_policy="spot",
#     environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
#     workspace="uwtacolab",
# )

cluster = coiled.Cluster(
    name="parquets_to_aggregation_netcdfs",
    idle_timeout="10 minutes",
    n_workers=50,
    worker_memory="16 GB",
    #worker_cpu=4,
    #scheduler_memory="128 GB",
    spot_policy="spot",
    environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
    workspace="uwtacolab",
)
client = cluster.get_client()

In [ ]:
client.restart()

In [ ]:
def process_and_save_geographic_unit(unit_id, unit_id_column, parquet_path, filesystem, 
                                   dem_bin_low=0, dem_bin_high=9000, 
                                   dem_bin_interval=100, aspect_bin_interval=15):
    """
    Generic function to process single geographic unit and return netCDF dataset
    
    Parameters:
    -----------
    unit_id : int
        ID of the geographic unit to process
    unit_id_column : str
        Column name containing the unit IDs (e.g., 'GMBA_V2_ID', 'PFAF_ID')
    parquet_path : str
        Path to the parquet dataset
    filesystem : filesystem object
        Filesystem to use for reading data
    """
    
    # Setup bins and coordinates
    dem_bins = np.arange(dem_bin_low, dem_bin_high+dem_bin_interval, dem_bin_interval)
    aspect_bins = np.arange(0, 360+aspect_bin_interval, aspect_bin_interval)
    water_years = range(2015, 2025)
    dem_coords = dem_bins[:-1] + dem_bin_interval/2
    aspect_coords = aspect_bins[:-1] + aspect_bin_interval/2
    stat_coords = ["mean", "median", "count"]

    # Read data
    cols_needed = [unit_id_column, 'dem', 'aspect', 'runoff_onset_median', 'runoff_onset_mad'] + \
                  [f'runoff_onset_WY{year}' for year in water_years]
    
    unit_ddf = dd.read_parquet(
        parquet_path,
        filesystem=filesystem,
        columns=cols_needed,
        filters=[(unit_id_column, '==', unit_id)]
    )

    # Create bins and calculate anomalies
    def bin_and_anomaly(df):
        df = df.dropna(subset=['dem', 'aspect'])
        df['dem_bin'] = pd.cut(df['dem'], dem_bins).apply(
            lambda x: x.left+(dem_bin_interval/2) if pd.notnull(x) else np.nan
        )
        df['aspect_bin'] = pd.cut(df['aspect'], aspect_bins).apply(
            lambda x: x.left+(aspect_bin_interval/2) if pd.notnull(x) else np.nan
        )
        df['dem_bin'] = df['dem_bin'].astype('Int64')
        df['aspect_bin'] = df['aspect_bin'].astype('Float64')
        
        # Calculate anomalies
        for year in water_years:
            col = f'runoff_onset_WY{year}'
            anom_col = f'runoff_onset_anomaly_WY{year}'
            df[anom_col] = df[col].where(df[col] > 0) - df['runoff_onset_median']
        
        return df

    unit_ddf = unit_ddf.map_partitions(bin_and_anomaly)
    unit_ddf = unit_ddf.dropna(subset=['dem_bin', 'aspect_bin'])
    unit_ddf = unit_ddf.compute()

    # Calculate static statistics
    agg_static = unit_ddf.groupby(['dem_bin', 'aspect_bin']).agg({
        'runoff_onset_median': ['mean', 'median', 'count'],
        'runoff_onset_mad': ['mean', 'median', 'count']
    })#.compute()

    # Calculate yearly aggregations
    yearly_aggs = []
    anomaly_aggs = []
    
    for year in water_years:
        year_col = f'runoff_onset_WY{year}'
        # Filter out invalid values
        year_data = unit_ddf[unit_ddf[year_col] > 0][['dem_bin', 'aspect_bin', year_col]]
        
        year_agg = year_data.groupby(['dem_bin', 'aspect_bin']).agg({
            year_col: ['mean', 'median', 'count']
        })#.compute()
        
        year_agg.columns = ['mean', 'median', 'count']
        year_agg = year_agg.reset_index()
        year_agg['water_year'] = year
        yearly_aggs.append(year_agg)

        # Process anomaly data
        anom_col = f'runoff_onset_anomaly_WY{year}'
        anom_data = unit_ddf.dropna(subset=[anom_col])[['dem_bin', 'aspect_bin', anom_col]]
        
        anom_agg = anom_data.groupby(['dem_bin', 'aspect_bin']).agg({
            anom_col: ['mean', 'median', 'count']
        })#.compute()
        
        anom_agg.columns = ['mean', 'median', 'count']
        anom_agg = anom_agg.reset_index()
        anom_agg['water_year'] = year
        anomaly_aggs.append(anom_agg)

    # Convert to pandas DataFrames
    yearly_aggs = pd.concat(yearly_aggs, ignore_index=True)
    anomaly_aggs = pd.concat(anomaly_aggs, ignore_index=True)
    
    # Create dataset
    ds = xr.Dataset(
        coords={
            'elevation': dem_coords,
            'aspect': aspect_coords,
            'statistic': stat_coords,
            'water_year': list(water_years)
        }
    )

    # Initialize arrays
    shape_static = (len(ds.elevation), len(ds.aspect), len(ds.statistic))
    shape_yearly = shape_static + (len(ds.water_year),)

    ds['runoff_onset_median'] = xr.DataArray(np.full(shape_static, np.nan), 
        dims=('elevation', 'aspect', 'statistic'))
    ds['runoff_onset_mad'] = xr.DataArray(np.full(shape_static, np.nan), 
        dims=('elevation', 'aspect', 'statistic'))
    ds['runoff_onset'] = xr.DataArray(np.full(shape_yearly, np.nan), 
        dims=('elevation', 'aspect', 'statistic', 'water_year'))
    ds['runoff_onset_anomaly'] = xr.DataArray(np.full(shape_yearly, np.nan), 
        dims=('elevation', 'aspect', 'statistic', 'water_year'))

    # Fill values using pandas indexing
    for dem in ds.elevation.values:
        for asp in ds.aspect.values:
            # Fill static variables
            static_mask = (agg_static.index.get_level_values('dem_bin') == dem) & \
                         (agg_static.index.get_level_values('aspect_bin') == asp)
            
            if static_mask.any():
                static_data = agg_static[static_mask]
                for stat in ['mean', 'median', 'count']:
                    ds['runoff_onset_median'].loc[{
                        'elevation': dem,
                        'aspect': asp,
                        'statistic': stat
                    }] = static_data[('runoff_onset_median', stat)].iloc[0]
                    
                    ds['runoff_onset_mad'].loc[{
                        'elevation': dem,
                        'aspect': asp,
                        'statistic': stat
                    }] = static_data[('runoff_onset_mad', stat)].iloc[0]
            
            # Fill yearly variables
            year_mask = (yearly_aggs['dem_bin'] == dem) & (yearly_aggs['aspect_bin'] == asp)
            anom_mask = (anomaly_aggs['dem_bin'] == dem) & (anomaly_aggs['aspect_bin'] == asp)
            
            for stat in ['mean', 'median', 'count']:
                year_data = yearly_aggs[year_mask]
                if not year_data.empty:
                    for _, row in year_data.iterrows():
                        ds['runoff_onset'].loc[{
                            'elevation': dem,
                            'aspect': asp,
                            'statistic': stat,
                            'water_year': row['water_year']
                        }] = row[stat]
                
                anom_data = anomaly_aggs[anom_mask]
                if not anom_data.empty:
                    for _, row in anom_data.iterrows():
                        ds['runoff_onset_anomaly'].loc[{
                            'elevation': dem,
                            'aspect': asp,
                            'statistic': stat,
                            'water_year': row['water_year']
                        }] = row[stat]

    # Add attributes
    ds.elevation.attrs['units'] = 'meters'
    ds.aspect.attrs['units'] = 'degrees'
    ds.runoff_onset.attrs['units'] = 'day of water year'
    ds.runoff_onset_anomaly.attrs['units'] = 'days'
    ds.runoff_onset_median.attrs['units'] = 'day of water year'
    ds.runoff_onset_mad.attrs['units'] = 'days'
    
    ds.attrs['location'] = unit_id
    ds.attrs['unit_type'] = unit_id_column

    return ds.compute()

In [ ]:
@dask.delayed
def get_unique_ids(parquet_path, filesystem, id_column):
    """Get unique IDs from parquet dataset"""
    unique_ids = dd.read_parquet(
        parquet_path, 
        filesystem=filesystem,
        columns=[id_column],
    )[id_column].unique().compute()
    return unique_ids
    
def process_all_units(parquet_path, output_dir, filesystem, unit_type, 
                     id_column, batch_size=30):
    """
    Process all geographic units in batches
    
    Parameters:
    -----------
    parquet_path : str
        Path to parquet dataset
    output_dir : str
        Directory to save output files
    filesystem : filesystem object
        Filesystem for reading data
    unit_type : str
        Type of unit ('mountain_range' or 'river_basin')
    id_column : str
        Column name for unit IDs
    batch_size : int
        Number of units to process per batch
    """
    os.makedirs(output_dir, exist_ok=True)

    # Get all unit IDs
    unit_ids = get_unique_ids(parquet_path, filesystem, id_column).compute()

    # remove unit id of -9999
    unit_ids = unit_ids[unit_ids != -9999]

    # Filter out already processed units
    unprocessed_units = [
        unit_id for unit_id in unit_ids 
        if not os.path.exists(f"{output_dir}/{unit_type}_{unit_id}.nc")
    ]

    print(f"Found {len(unprocessed_units)} unprocessed {unit_type}s")

    # Process in batches
    for i in range(0, len(unprocessed_units), batch_size):
        batch = unprocessed_units[i:i + batch_size]
        futures = []

        print(f"Processing batch {i//batch_size + 1} of {(len(unprocessed_units)-1)//batch_size + 1}")
        
        # Submit batch of tasks
        for unit_id in batch:
            future = client.submit(
                process_and_save_geographic_unit, 
                unit_id, id_column, parquet_path, filesystem
            )
            futures.append(future)

        # Wait for batch completion and save results
        for future, result in dask.distributed.as_completed(futures, with_results=True):
            unit_id = result.attrs['location']
            output_file = f"{output_dir}/{unit_type}_{unit_id}.nc"
            result.to_netcdf(output_file)
            print(f"Saved to {output_file}")

        # Restart client between batches to clear memory
        client.restart()

In [ ]:
DATASETS = {
    'fcf_lte_50': f'snowmelt/analysis/parquets/full_datasets/fcf_lte_50/{config.version}',
    #'full_dataset': f'snowmelt/analysis/parquets/full_datasets/full_dataset/{config.version}', 
    #'no_trees': f'snowmelt/analysis/parquets/full_datasets/no_trees/{config.version}'
}

UNIT_CONFIGS = {
    'mountain_ranges': {
        'id_column': 'GMBA_V2_ID',
        'output_prefix': 'mountain_range'
    },
    'river_basins': {
        'id_column': 'PFAF_ID', 
        'output_prefix': 'river_basin'
    }
}

In [ ]:
client.restart()

## mountain ranges

In [ ]:
# print("Processing Mountain Ranges...")
# for dataset_name, base_path in DATASETS.items():
#     print(f"\n=== Processing {dataset_name} ===")
    
#     parquet_path = base_path
#     output_dir = f"aggregated_results/mountain_ranges/{dataset_name}/{config.version}"
    
#     client.restart()
#     process_all_units(
#         parquet_path=parquet_path,
#         output_dir=output_dir, 
#         filesystem=config.azure_blob_fs,
#         unit_type='mountain_range',
#         id_column='GMBA_V2_ID',
#         batch_size=20
#     )

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)

def merge_mountain_range_netcdfs(netcdf_dir):
    from collections import Counter

    """Merge all mountain range netCDFs"""
    nc_files = sorted(glob.glob(f"{netcdf_dir}/mountain_range_*.nc"))
    datasets = []
    
    for nc_file in nc_files:
        ds = xr.open_dataset(nc_file,decode_times=False)
        mr_id = int(nc_file.split('_')[-1].split('.')[0])
        ds = ds.expand_dims({'mountain_range': [mr_id]})
        datasets.append(ds)
    
    merged_ds = xr.concat(datasets, dim='mountain_range')


    merged_ds['aspect'] = np.deg2rad(merged_ds['aspect'])
    merged_ds['aspect'].attrs['units'] = 'radians'

    merged_ds = merged_ds.sel(mountain_range=merged_ds.mountain_range != -9999)

    # Create a dictionary mapping from GMBA_V2_ID to MapName
    id_to_name = dict(zip(gmba_gdf['GMBA_V2_ID'], gmba_gdf['MapName']))
    id_to_level04 = dict(zip(gmba_gdf['GMBA_V2_ID'], gmba_gdf['Level_04']))

    mr_ids = merged_ds.mountain_range.values
    mr_names = [id_to_name[x] for x in mr_ids]

    name_counts = Counter(mr_names)
    duplicated_names = {name for name, count in name_counts.items() if count > 1}

    final_names = []

    for mr_id, name in zip(mr_ids, mr_names):
        if name in duplicated_names:
            # Use Level_04 for ALL instances of duplicated names
            final_names.append(id_to_level04[mr_id])
        else:
            final_names.append(name)
            
    # Sort the dataset by the final names
    merged_ds = merged_ds.assign_coords(mountain_range=final_names)

    # merged_ds = merged_ds.assign_coords(
    #     mountain_range=[id_to_name[x] for x in merged_ds.mountain_range.values]
    # )

    merged_ds = merged_ds.sortby('mountain_range')

    merged_ds['runoff_onset_elev_relative'] = merged_ds['runoff_onset_median'] - merged_ds['runoff_onset_median'].median(dim='aspect')
    merged_ds['runoff_onset_elev_relative'].loc[{'statistic': 'count'}] = merged_ds['runoff_onset_median'].sel(statistic='count')
    
    return merged_ds

In [ ]:
#binned_ds = merge_mountain_range_netcdfs(netcdf_dir="mountain_ranges/full_dataset")
#binned_ds = merge_mountain_range_netcdfs(netcdf_dir="aggregated_results/mountain_ranges/fcf_lte_50")
binned_ds = merge_mountain_range_netcdfs(netcdf_dir="aggregated_results/mountain_ranges/fcf_lte_50/v9")

#binned_ds = merge_mountain_range_netcdfs(netcdf_dir="aggregated_results/mountain_ranges/no_trees")

binned_ds

In [ ]:
# Load and prepare world data
continents_gdf = gpd.read_file(f"zip+https://pubs.usgs.gov/of/2006/1187/basemaps/continents/continents.zip")

# Project GMBA for accurate centroids
projected_gmba = gmba_gdf.to_crs("EPSG:3857")

# Create mountain points with projected centroids
mountain_data = []
for name in binned_ds.mountain_range.values:
    if name in gmba_gdf['MapName'].values:
        point_data = projected_gmba[projected_gmba['MapName'] == name]
    else:
        point_data = projected_gmba[projected_gmba['Level_04'] == name]
    
    centroid = point_data.geometry.centroid.iloc[0]
    centroid_wgs84 = gpd.GeoSeries([Point(centroid.x, centroid.y)], crs="EPSG:3857").to_crs("EPSG:4326")
    
    mountain_data.append({
        'name': name,
        'geometry': centroid_wgs84.iloc[0],
        'latitude': centroid.y,
        'longitude': centroid.x
    })

mountain_points = gpd.GeoDataFrame(mountain_data)
mountain_points = mountain_points.set_geometry('geometry').set_crs("EPSG:4326")
range_metadata = gpd.sjoin_nearest(mountain_points, continents_gdf[['CONTINENT', 'geometry']], how='left')
range_metadata = range_metadata.replace(to_replace='Australia', value='Oceania')
range_metadata

In [ ]:
# now add lat/lon and continent to binned_ds
binned_ds = binned_ds.assign_coords(
    centroid_latitude=('mountain_range', range_metadata['geometry'].y.values),
    centroid_longitude=('mountain_range', range_metadata['geometry'].x.values),
    continent=('mountain_range', range_metadata['CONTINENT'].values)
)
binned_ds

In [ ]:
era5_anomaly_ds = xr.open_zarr(config.azure_blob_fs.get_mapper(f"snowmelt/analysis/era5_data/era5_land_anomaly_ds.zarr"),decode_coords='all', chunks="auto")
era5_anomaly_ds

# using this one instead because we masked era5 by runoff_onset for each water year. this way we are only looking at era5 pixels that affect runoff onset anomaly
# era5_anomaly_ds = xr.open_zarr(config.azure_blob_fs.get_mapper(f"snowmelt/analysis/era5_data/combined_runoff_onset_and_era5_eqearth_anomaly_ds.zarr"),decode_coords='all', chunks="auto")
# era5_anomaly_ds

In [ ]:
cluster = coiled.Cluster(
    name="compare-runoff-anomaly-to-era5",
    idle_timeout="10 minutes",
    n_workers=2,
    worker_memory="256 GB",
    #worker_cpu=4,
    #scheduler_memory="128 GB",
    spot_policy="spot",
    environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
    workspace="uwtacolab",
)
client = cluster.get_client()

In [ ]:
# mountain_ranges = binned_ds['mountain_range'].values

# def add_era5_anomaly_to_mountain_ranges(config, mountain_ranges):
    
#     mountain_datasets = []
    
#     era5_anomaly_ds = xr.open_zarr(config.azure_blob_fs.get_mapper(f"snowmelt/analysis/era5_data/era5_land_anomaly_ds.zarr"),decode_coords='all', chunks="auto")
    
#     url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
#     gmba_gdf = gpd.read_file("zip+" + url)


#     for mountain_range in mountain_ranges:
#         print(mountain_range)
        
#         if mountain_range in gmba_gdf['MapName'].values:
#             mountain_gdf = gmba_gdf[gmba_gdf['MapName'] == mountain_range]
#         else:
#             mountain_gdf = gmba_gdf[gmba_gdf['Level_04'] == mountain_range]
            
#         utm_proj = mountain_gdf.estimate_utm_crs()
#         mountain_utm_gdf = mountain_gdf.to_crs(utm_proj)
        
#         snow_class_da = easysnowdata.remote_sensing.get_seasonal_snow_classification(bbox_input=mountain_gdf)
#         snow_class_proj_da = snow_class_da.rio.reproject(utm_proj, resampling=rasterio.enums.Resampling.mode, resolution=1000)
        
#         #era5_land_spring_temp_10yr_anomaly_clipped_da = #.rio.clip_box(*mountain_utm_gdf.total_bounds, crs=mountain_utm_gdf.crs)
#         #mountain_era5_anomaly_clipbox_ds = era5_anomaly_ds.rio.clip_box(*mountain_utm_gdf.total_bounds, crs=mountain_utm_gdf.crs).compute()
#         mountain_era5_anomaly_ds = era5_anomaly_ds.rio.clip(mountain_utm_gdf.geometry.values, crs=mountain_utm_gdf.crs).compute()

#         mountain_era5_anomaly_utm_ds = mountain_era5_anomaly_ds.odc.reproject(snow_class_proj_da.odc.geobox)
        
#         store = config.azure_blob_fs.get_mapper(f"snowmelt/snowmelt_runoff_onset/coarsened/global_v9_coarsened_20_ds.zarr")
#         runoff_onset_ds = xr.open_zarr(store, consolidated=True, decode_coords='all', chunks="auto").rio.write_crs("EPSG:4326")
        
#         runoff_onset_ds = runoff_onset_ds.rio.clip_box(*mountain_gdf.total_bounds, crs="EPSG:4326")
#         mountain_runoff_onset_proj_ds = runoff_onset_ds.rio.clip(mountain_utm_gdf.geometry.values, crs=mountain_utm_gdf.crs).rio.reproject_match(mountain_era5_anomaly_utm_ds)
        
#         mountain_era5_anomaly_utm_ds = mountain_era5_anomaly_utm_ds.where(mountain_runoff_onset_proj_ds['runoff_onset'].notnull())
#         mountain_era5_anomaly_utm_ds = mountain_era5_anomaly_utm_ds.where(snow_class_proj_da!=4)

#         #mountain_spring_temp_10yr_anomaly_utm_da = era5_land_spring_temp_10yr_anomaly_utm_da.rio.clip(mountain_utm_gdf.geometry, crs=mountain_utm_gdf.crs)
#         mountain_mean_era5_anomaly_ds = mountain_era5_anomaly_utm_ds.mean(dim=['x','y']).expand_dims({'mountain_range':[mountain_range]}).drop_vars('spatial_ref')
        
#         mountain_datasets.append(mountain_mean_era5_anomaly_ds)
        
#     return mountain_datasets
    

In [ ]:
def process_single_mountain_range(config, mountain_range):
    """Process a single mountain range and return its ERA5 anomaly dataset"""
    
    era5_anomaly_ds = xr.open_zarr(config.azure_blob_fs.get_mapper(f"snowmelt/analysis/era5_data/era5_land_anomaly_ds.zarr"),decode_coords='all', chunks="auto")
    
    url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
    gmba_gdf = gpd.read_file("zip+" + url)

    print(f"Processing {mountain_range}")
    
    if mountain_range in gmba_gdf['MapName'].values:
        mountain_gdf = gmba_gdf[gmba_gdf['MapName'] == mountain_range]
    else:
        mountain_gdf = gmba_gdf[gmba_gdf['Level_04'] == mountain_range]
        
    # if Aleutians, clip to -178
    if mountain_range == "Aleutian Ranges":
        mountain_gdf = mountain_gdf.clip((-179.9,0,-100,90),)
        
    if mountain_range == 'Central Range':
        mountain_gdf = mountain_gdf.clip((120.9,-20,155,10),)
        
    if mountain_range == 'Arctic Ocean':
        mountain_gdf = mountain_gdf.clip((-179.9,76,179.9,89.9),)
        
    if mountain_range == 'South Atlantic Islands':
        mountain_gdf = mountain_gdf.clip((-62,-55,-58,-48),)



    utm_proj = mountain_gdf.estimate_utm_crs()
    mountain_utm_gdf = mountain_gdf.to_crs(utm_proj)
    
    snow_class_da = easysnowdata.remote_sensing.get_seasonal_snow_classification(bbox_input=mountain_gdf)
    snow_class_proj_da = snow_class_da.rio.reproject(utm_proj, resampling=rasterio.enums.Resampling.mode, resolution=1000)
    
    mountain_era5_anomaly_ds = era5_anomaly_ds.rio.clip(mountain_utm_gdf.geometry.values, crs=mountain_utm_gdf.crs).compute()
    mountain_era5_anomaly_utm_ds = mountain_era5_anomaly_ds.odc.reproject(snow_class_proj_da.odc.geobox)
    
    store = config.azure_blob_fs.get_mapper(f"snowmelt/snowmelt_runoff_onset/coarsened/global_v9_coarsened_20_ds.zarr")
    runoff_onset_ds = xr.open_zarr(store, consolidated=True, decode_coords='all', chunks="auto").rio.write_crs("EPSG:4326")
    
    runoff_onset_ds = runoff_onset_ds.rio.clip_box(*mountain_gdf.total_bounds, crs="EPSG:4326")
    mountain_runoff_onset_proj_ds = runoff_onset_ds.rio.clip(mountain_utm_gdf.geometry.values, crs=mountain_utm_gdf.crs).rio.reproject_match(mountain_era5_anomaly_utm_ds)
    
    mountain_era5_anomaly_utm_ds = mountain_era5_anomaly_utm_ds.where(mountain_runoff_onset_proj_ds['runoff_onset'].notnull())
    mountain_era5_anomaly_utm_ds = mountain_era5_anomaly_utm_ds.where(snow_class_proj_da!=4)

    mountain_mean_era5_anomaly_ds = mountain_era5_anomaly_utm_ds.mean(dim=['x','y']).expand_dims({'mountain_range':[mountain_range]}).drop_vars('spatial_ref')
    
    return mountain_mean_era5_anomaly_ds

def add_era5_anomaly_to_mountain_ranges_sequential(config, mountain_ranges):
    """Process mountain ranges one at a time using client.submit()"""
    
    mountain_datasets = []
    
    for mountain_range in mountain_ranges:
        print(f"Submitting job for {mountain_range}")
        # time this loop
        start_time = time.time()
        
        # Submit the job for this mountain range
        future = client.submit(process_single_mountain_range, config, mountain_range)
        
        # Wait for the result
        result = future.result()
        
        # Append to our list
        mountain_datasets.append(result)
        end_time = time.time()
        
        print(f"Completed {mountain_range} in {end_time-start_time:.2f} seconds")
    
    return mountain_datasets

In [ ]:
# We should Skip South Atlantic Islands

In [ ]:
mountain_datasets = add_era5_anomaly_to_mountain_ranges_sequential(config, mountain_ranges)
mountain_datasets

In [ ]:
# mountain_datasets = []

# for mountain_range in binned_ds['mountain_range'].values:
#     print(mountain_range)
    
#     if mountain_range in gmba_gdf['MapName'].values:
#         mountain_gdf = gmba_gdf[gmba_gdf['MapName'] == mountain_range]
#     else:
#         mountain_gdf = gmba_gdf[gmba_gdf['Level_04'] == mountain_range]
        
#     utm_proj = mountain_gdf.estimate_utm_crs()
#     mountain_utm_gdf = mountain_gdf.to_crs(utm_proj)
    
#     #era5_land_spring_temp_10yr_anomaly_clipped_da = #.rio.clip_box(*mountain_utm_gdf.total_bounds, crs=mountain_utm_gdf.crs)
#     #mountain_era5_anomaly_clipbox_ds = era5_anomaly_ds.rio.clip_box(*mountain_utm_gdf.total_bounds, crs=mountain_utm_gdf.crs)
#     mountain_era5_anomaly_ds = era5_anomaly_ds.rio.clip(mountain_utm_gdf.geometry.values, crs=mountain_utm_gdf.crs).compute()

#     mountain_era5_anomaly_utm_ds = mountain_era5_anomaly_ds.odc.reproject(utm_proj)
    
    

#     #mountain_spring_temp_10yr_anomaly_utm_da = era5_land_spring_temp_10yr_anomaly_utm_da.rio.clip(mountain_utm_gdf.geometry, crs=mountain_utm_gdf.crs)
#     mountain_mean_era5_anomaly_ds = mountain_era5_anomaly_utm_ds.mean(dim=['x','y']).expand_dims({'mountain_range':[mountain_range]}).drop_vars('spatial_ref')
    
#     mountain_datasets.append(mountain_mean_era5_anomaly_ds)

In [ ]:
mountains_era5_anomalies_ds = xr.merge(mountain_datasets)
mountains_era5_anomalies_ds

In [ ]:
mountains_ds = xr.merge([binned_ds, mountains_era5_anomalies_ds])#.drop_vars(['runoff_onset','temporal_resolution'])])
mountains_ds

In [ ]:
mountains_ds.to_netcdf(f"aggregated_results/mountain_ranges/fcf_lte_50/{config.version}/all_mountain_ranges.nc")

## river basins

In [ ]:
# print("Processing River Basins...")
# for dataset_name, base_path in DATASETS.items():
#     print(f"\n=== Processing {dataset_name} ===")

#     parquet_path = base_path  
#     output_dir = f"aggregated_results/river_basins/{dataset_name}/{config.version}"

#     client.restart()
#     process_all_units(
#         parquet_path=parquet_path,
#         output_dir=output_dir,
#         filesystem=config.azure_blob_fs, 
#         unit_type='river_basin',
#         id_column='PFAF_ID',
#         batch_size=30
#     )

In [ ]:
def merge_river_basin_netcdfs(netcdf_dir):

    """Merge all river basin netCDFs"""
    nc_files = sorted(glob.glob(f"{netcdf_dir}/river_basin_*.nc")) # _4* for asia
    datasets = []
    
    for nc_file in nc_files:
        ds = xr.open_dataset(nc_file,decode_times=False)
        rb_id = int(nc_file.split('_')[-1].split('.')[0])
        ds = ds.expand_dims({'river_basin': [rb_id]})
        datasets.append(ds)
    
    merged_ds = xr.concat(datasets, dim='river_basin')


    merged_ds['aspect'] = np.deg2rad(merged_ds['aspect'])
    merged_ds['aspect'].attrs['units'] = 'radians'

    merged_ds = merged_ds.sel(river_basin=merged_ds.river_basin != -9999)


    merged_ds = merged_ds.sortby('river_basin')

    merged_ds['runoff_onset_elev_relative'] = merged_ds['runoff_onset_median'] - merged_ds['runoff_onset_median'].median(dim='aspect')
    merged_ds['runoff_onset_elev_relative'].loc[{'statistic': 'count'}] = merged_ds['runoff_onset_median'].sel(statistic='count')
    
    return merged_ds

In [ ]:
netcdf_dir = "aggregated_results/river_basins/v5_basins/fcf_lte_50"
netcdf_dir = "aggregated_results/river_basins/fcf_lte_50/v9"

merged_ds = merge_river_basin_netcdfs(netcdf_dir)
merged_ds

In [ ]:
# create weighted average of variables in each basin, weighted by count of samples in each bin. should create a basin_mean and basin_count variable
total_count_ds = merged_ds.sel(statistic='count').sum(dim=['elevation','aspect'])

basin_mean_ds = (merged_ds.sel(statistic='count')*merged_ds.sel(statistic='mean')).sum(dim=['elevation','aspect'])/(total_count_ds)

weighted_mean_ds = xr.concat([basin_mean_ds, total_count_ds], 
                           dim=pd.Index(['basin_mean', 'basin_count'], name='statistic'))

weighted_mean_ds = weighted_mean_ds.rename({var: f"basin_{var}" for var in weighted_mean_ds.data_vars})

weighted_mean_ds

In [ ]:
all_river_basins_ds = xr.merge([merged_ds, weighted_mean_ds])
all_river_basins_ds

In [ ]:
all_river_basins_ds['statistic'] = all_river_basins_ds['statistic'].astype(str)

In [ ]:
all_river_basins_ds.to_netcdf(f"aggregated_results/river_basins/fcf_lte_50/{config.version}/all_river_basins.nc")

## continents

In [ ]:
cluster = coiled.Cluster(
    name="parquets_to_aggregation_netcdfs",
    idle_timeout="10 minutes",
    n_workers=60,
    worker_memory="32 GB",
    worker_cpu=4,
    #scheduler_memory="128 GB",
    spot_policy="spot",
    environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
    workspace="uwtacolab",
)
client = cluster.get_client()

In [ ]:
client.restart()

In [ ]:
def create_lat_and_elev_binned_ds(continent_ddf, lat_bin_low=-80, lat_bin_high=80, 
                                  lat_bin_interval=1, dem_bin_low=0, dem_bin_high=9000, 
                                  dem_bin_interval=100):
    """
    Create comprehensive latitude and elevation binned dataset with:
    - Time series data (yearly runoff onset + anomalies)
    - CHILI analysis (cool/neutral/warm classes + correlations)
    - FCF correlation analysis
    - Static statistics (median, MAD)
    
    Keeps data as Dask DataFrame longer for memory efficiency with large continents
    """

    # Create bins
    dem_bins = np.arange(dem_bin_low, dem_bin_high+dem_bin_interval, dem_bin_interval)
    lat_bins = np.arange(lat_bin_low, lat_bin_high+lat_bin_interval, lat_bin_interval)

    continent_ddf['lat_bin'] = continent_ddf['original_lat'].map_partitions(pd.cut, lat_bins)
    continent_ddf['dem_bin'] = continent_ddf['dem'].map_partitions(pd.cut, dem_bins)
    continent_ddf = continent_ddf.dropna(subset=['lat_bin','dem_bin'])

    continent_ddf['lat_bin'] = continent_ddf['lat_bin'].apply(lambda x: x.left, meta=('lat_bin', 'int16')).astype('int16')
    continent_ddf['dem_bin'] = continent_ddf['dem_bin'].apply(lambda x: x.left, meta=('dem_bin', 'int16')).astype('int16')

    # Setup coordinates
    stat_coords = ["mean", "median", "count"]
    lat_coords = lat_bins[:-1] + lat_bin_interval / 2
    elev_coords = dem_bins[:-1] + dem_bin_interval / 2
    water_years = range(2015, 2025)

    # Initialize empty arrays with NaN values
    shape_static = (len(lat_coords), len(elev_coords), len(stat_coords))
    shape_yearly = shape_static + (len(water_years),)
    
    runoff_onset_array = np.full(shape_static, np.nan)
    mad_array = np.full(shape_static, np.nan)
    chili_cool_array = np.full(shape_static, np.nan)
    chili_neutral_array = np.full(shape_static, np.nan)
    chili_warm_array = np.full(shape_static, np.nan)
    chili_corr_array = np.full((len(lat_coords), len(elev_coords)), np.nan)
    fcf_corr_array = np.full((len(lat_coords), len(elev_coords)), np.nan)
    
    # Time series arrays
    runoff_onset_yearly_array = np.full(shape_yearly, np.nan)
    runoff_onset_anomaly_array = np.full(shape_yearly, np.nan)

    # Create indexing dictionaries - match bins to coordinate indices
    # lat_bins[:-1] are the left edges (integers), lat_coords are centers (floats)
    lat_bin_to_idx = {int(bin_edge): i for i, bin_edge in enumerate(lat_bins[:-1])}
    elev_bin_to_idx = {int(bin_edge): i for i, bin_edge in enumerate(dem_bins[:-1])}
    stat_idx = {stat: i for i, stat in enumerate(stat_coords)}

    # CHILI classification - keep as Dask operations for memory efficiency
    continent_ddf["chili_class"] = "neutral"
    continent_ddf["chili_class"] = continent_ddf["chili_class"].where(
        (continent_ddf["chili"] >= 0.448) & (continent_ddf["chili"] <= 0.767),
        other=continent_ddf["chili"].map(
            lambda x: "warm" if x > 0.767 else "cool" if x < 0.448 else "neutral"
        ),
    )

    continent_ddf = continent_ddf.persist()

    # Process yearly data and anomalies - keep as Dask operations
    yearly_aggs = []
    anomaly_aggs = []
    
    # Filter negative values for each year and calculate anomalies
    for year in water_years:
        year_col = f'runoff_onset_WY{year}'
        anom_col = f'anomaly_WY{year}'
        
        # Filter out invalid values and calculate anomalies
        continent_ddf[year_col] = continent_ddf[year_col].where(continent_ddf[year_col] > 0)
        continent_ddf[anom_col] = continent_ddf[year_col] - continent_ddf['runoff_onset_median']

    # Perform all Dask computations at once
    #with dask.config.set({"dataframe.shuffle.method": "tasks"}): # usually tasks, trying disk to see if asia works now
        # Static aggregations
    agg_static = continent_ddf.groupby(["lat_bin", "dem_bin"]).aggregate({
        "runoff_onset_median": ["mean", "median", "count"],
        "runoff_onset_mad": ["mean", "median", "count"],
    }).compute()

    # CHILI class aggregations
    chili_and_median_runoff_onset_groupby_df = (
        continent_ddf[["lat_bin", "dem_bin", "chili_class", "runoff_onset_median"]]
        .dropna()
        .groupby(["lat_bin", "dem_bin", "chili_class"])["runoff_onset_median"]
        .agg(["mean", "median", "count"])
        .compute()
    )

    # CHILI correlation analysis
    chili_corr_groupby_df = (
        continent_ddf[["lat_bin", "dem_bin", "chili", "runoff_onset_median"]]
        .dropna()
        .groupby(["lat_bin", "dem_bin"])
        .apply(lambda x: x["chili"].corr(x["runoff_onset_median"]))
        .compute()
    )

    # FCF correlation analysis
    fcf_corr_groupby_df = (
        continent_ddf[["lat_bin", "dem_bin", "forest_cover_fraction", "runoff_onset_median"]]
        .dropna()
        .groupby(["lat_bin", "dem_bin"])
        .apply(lambda x: x["forest_cover_fraction"].corr(x["runoff_onset_median"]))
        .compute()
    )

    # Yearly aggregations
    for year in water_years:
        year_col = f'runoff_onset_WY{year}'
        anom_col = f'anomaly_WY{year}'
        
        year_agg = continent_ddf.groupby(['lat_bin', 'dem_bin'])[year_col].agg(['mean', 'median', 'count']).compute()
        year_agg.columns = ['mean', 'median', 'count']
        year_agg = year_agg.reset_index()
        year_agg['water_year'] = year
        yearly_aggs.append(year_agg)
        
        anom_agg = continent_ddf.groupby(['lat_bin', 'dem_bin'])[anom_col].agg(['mean', 'median', 'count']).compute()
        anom_agg.columns = ['mean', 'median', 'count']
        anom_agg = anom_agg.reset_index()
        anom_agg['water_year'] = year
        anomaly_aggs.append(anom_agg)

    # Convert yearly aggregations to pandas DataFrames
    yearly_aggs = pd.concat(yearly_aggs, ignore_index=True)
    anomaly_aggs = pd.concat(anomaly_aggs, ignore_index=True)

    # Fill static arrays
    for (lat, elev), row in agg_static.iterrows():
        if lat in lat_bin_to_idx and elev in elev_bin_to_idx:
            i, j = lat_bin_to_idx[lat], elev_bin_to_idx[elev]
            for stat in stat_coords:
                k = stat_idx[stat]
                runoff_onset_array[i, j, k] = row[("runoff_onset_median", stat)]
                mad_array[i, j, k] = row[("runoff_onset_mad", stat)]

    # Fill CHILI class arrays
    for (lat, elev, chili_class), row in chili_and_median_runoff_onset_groupby_df.iterrows():
        if lat in lat_bin_to_idx and elev in elev_bin_to_idx:
            i, j = lat_bin_to_idx[lat], elev_bin_to_idx[elev]
            target_array = {
                "cool": chili_cool_array,
                "neutral": chili_neutral_array,
                "warm": chili_warm_array,
            }[chili_class]
            for stat in stat_coords:
                k = stat_idx[stat]
                target_array[i, j, k] = row[("runoff_onset_median", stat)]

    # Fill correlation arrays
    for (lat, elev), corr in chili_corr_groupby_df.items():
        if lat in lat_bin_to_idx and elev in elev_bin_to_idx:
            i, j = lat_bin_to_idx[lat], elev_bin_to_idx[elev]
            chili_corr_array[i, j] = corr

    for (lat, elev), corr in fcf_corr_groupby_df.items():
        if lat in lat_bin_to_idx and elev in elev_bin_to_idx:
            i, j = lat_bin_to_idx[lat], elev_bin_to_idx[elev]
            fcf_corr_array[i, j] = corr

    # Fill yearly time series arrays
    for _, row in yearly_aggs.iterrows():
        lat, elev = row['lat_bin'], row['dem_bin']
        if lat in lat_bin_to_idx and elev in elev_bin_to_idx:
            i, j = lat_bin_to_idx[lat], elev_bin_to_idx[elev]
            year_idx = list(water_years).index(row['water_year'])
            for stat in stat_coords:
                k = stat_idx[stat]
                runoff_onset_yearly_array[i, j, k, year_idx] = row[stat]

    for _, row in anomaly_aggs.iterrows():
        lat, elev = row['lat_bin'], row['dem_bin']
        if lat in lat_bin_to_idx and elev in elev_bin_to_idx:
            i, j = lat_bin_to_idx[lat], elev_bin_to_idx[elev]
            year_idx = list(water_years).index(row['water_year'])
            for stat in stat_coords:
                k = stat_idx[stat]
                runoff_onset_anomaly_array[i, j, k, year_idx] = row[stat]

    # Create comprehensive xarray Dataset
    ds = xr.Dataset(
        data_vars={
            # Static variables
            "runoff_onset_median": (("latitude", "elevation", "statistic"), runoff_onset_array),
            "runoff_onset_mad": (("latitude", "elevation", "statistic"), mad_array),
            
            # CHILI analysis
            "chili_cool": (("latitude", "elevation", "statistic"), chili_cool_array),
            "chili_neutral": (("latitude", "elevation", "statistic"), chili_neutral_array),
            "chili_warm": (("latitude", "elevation", "statistic"), chili_warm_array),
            "chili_corr": (("latitude", "elevation"), chili_corr_array),
            
            # FCF analysis
            "fcf_corr": (("latitude", "elevation"), fcf_corr_array),
            
            # Time series variables
            "runoff_onset": (("latitude", "elevation", "statistic", "water_year"), runoff_onset_yearly_array),
            "runoff_onset_anomaly": (("latitude", "elevation", "statistic", "water_year"), runoff_onset_anomaly_array),
        },
        coords={
            "latitude": lat_coords,
            "elevation": elev_coords,
            "statistic": stat_coords,
            "water_year": list(water_years),
        },
    )

    # Calculate derived CHILI variables
    ds["chili_warm_cool_ratio"] = ds["chili_warm"] / ds["chili_cool"]
    ds["chili_warm_cool_ratio"].loc[{"statistic": "count"}] = ds["chili_warm"].sel(statistic="count")

    ds["chili_warm_cool_difference"] = ds["chili_warm"] - ds["chili_cool"]
    ds["chili_warm_cool_difference"].loc[{"statistic": "count"}] = ds["chili_warm"].sel(statistic="count")

    # Add attributes
    ds.latitude.attrs['units'] = 'degrees'
    ds.elevation.attrs['units'] = 'meters'

    return ds

In [ ]:
def process_continent(parquet_path, filesystem, continent):
    """
    Process single continent with all analysis (static + time series + CHILI + FCF)
    """
    continents_enum = {
        0: "Africa", 1: "Antarctica", 2: "Asia", 3: "Australia",
        4: "Europe", 5: "North America", 6: "Oceania", 7: "South America",
    }
    
    if continent != "Oceania":
        continent_id = list(continents_enum.keys())[list(continents_enum.values()).index(continent)]
        continent_ddf = dd.read_parquet(
            parquet_path,
            filesystem=filesystem,
            filters=[('continent', '==', continent_id)]
        )
    else:
        # Oceania combines Australia and Oceania
        continent_ddf = dd.read_parquet(
            parquet_path,
            filesystem=filesystem,
            filters=[('continent', 'in', [3, 6])]
        )

    ds = create_lat_and_elev_binned_ds(continent_ddf)
    ds.attrs['location'] = continent
    ds.attrs['processing_type'] = 'continent'

    return ds.compute()


In [ ]:
def process_all_continents(parquet_path, output_dir, filesystem):
    """Process all continents with comprehensive analysis"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    continents = ['Africa', 'Asia', 'Europe', 'Oceania', 'North America', 'South America']

    unprocessed_continents = [
        continent for continent in continents 
        if not os.path.exists(f"{output_dir}/continent_{continent.replace(' ', '_')}.nc")
    ]
    
    for continent in unprocessed_continents:
        print(f"Continent {continent} is unprocessed.")
    
        
    # Submit processing jobs
    for continent in unprocessed_continents:
        print(f"Processing continent: {continent}")
        
        future = client.submit(process_continent, parquet_path, filesystem, continent)

        result = future.result()
        
        output_file = f"{output_dir}/continent_{continent.replace(' ', '_')}.nc"
        result.to_netcdf(output_file)
        print(f"Saved to {output_file}")


In [ ]:
process_all_continents(
    parquet_path=f'snowmelt/analysis/parquets/full_datasets/fcf_lte_50/{config.version}',
    output_dir=f'aggregated_results/continents/fcf_lte_50/{config.version}',
    filesystem=config.azure_blob_fs
)

In [ ]:
client.restart()

In [ ]:
def merge_continent_netcdfs(netcdf_dir):

    nc_files = sorted(glob.glob(f"{netcdf_dir}/continent_*.nc"))
    datasets = []
    for nc_file in nc_files:
        ds = xr.open_dataset(nc_file,decode_times=False)
        continent_name = nc_file.split('continent_')[-1].split('.')[0]
        if '_' in continent_name:
            continent_name = continent_name.replace('_', ' ')
        ds = ds.expand_dims({'continent': [continent_name]})
        datasets.append(ds)

    merged_ds = xr.concat(datasets, dim='continent')


    merged_ds['fcf_corr'] = merged_ds['fcf_corr'].expand_dims(
    {"statistic": ['count', 'mean', 'std']}, axis=0
    )
    merged_ds['fcf_corr'].loc[{"statistic": "count"}] = merged_ds['runoff_onset_median'].loc[{"statistic": "count"}]

    merged_ds['chili_corr'] = merged_ds['chili_corr'].expand_dims(
        {"statistic": ['count', 'mean', 'std']}, axis=0
    )
    merged_ds['chili_corr'].loc[{"statistic": "count"}] = merged_ds['runoff_onset_median'].loc[{"statistic": "count"}]

    return merged_ds

In [ ]:
merged_ds = merge_continent_netcdfs(
    netcdf_dir=f'aggregated_results/continents/fcf_lte_50/{config.version}'
)
merged_ds

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import ee

ee.Initialize(project="egagli-data-access", opt_url="https://earthengine-highvolume.googleapis.com")

# Load elevation data from GTOPO30
dem = ee.Image("USGS/GTOPO30")

# Load continent boundaries
continents_gdf = gpd.read_file("zip+https://pubs.usgs.gov/of/2006/1187/basemaps/continents/continents.zip")
continents = ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America']

# Define the bins
lat_bin_low = -90
lat_bin_high = 90  
lat_bin_interval = 1
lat_bins = np.arange(lat_bin_low, lat_bin_high + lat_bin_interval, lat_bin_interval)

dem_bin_low = 0
dem_bin_high = 9000  # Max elevation on Earth is ~8850m
dem_bin_interval = 100
dem_bins = np.arange(dem_bin_low, dem_bin_high + dem_bin_interval, dem_bin_interval)

# Create the coordinates for our xarray
lat_coords = lat_bins[:-1] + lat_bin_interval/2  # Use centers of bins
dem_coords = dem_bins[:-1] + dem_bin_interval/2

# Initialize the dataset with zeros
hist_data = np.zeros((len(continents), len(lat_coords), len(dem_coords)), dtype=np.int64)

# Function to compute 2D histogram for a continent
def compute_continent_histogram(continent_name):
    # Get continent geometry from GeoDataFrame
    # Get continent geometry from GeoDataFrame
    if continent_name == 'Oceania':
        # For Oceania, include both Oceania and Australia
        continent_geom = continents_gdf[continents_gdf.CONTINENT.isin(['Oceania', 'Australia'])]
    else:
        continent_geom = continents_gdf[continents_gdf.CONTINENT == continent_name]
    
    
    if continent_geom.empty:
        print(f"No geometry found for {continent_name}")
        return np.zeros((len(lat_coords), len(dem_coords)))
    
    # Convert continent geometry to Earth Engine feature
    continent_ee = ee.FeatureCollection(continent_geom.__geo_interface__)
    
    # Create a latitude image for binning
    latitudes = ee.Image.pixelLonLat().select('latitude')
    
    lat_binned = latitudes.add(90).divide(lat_bin_interval).floor().int()
    
    # Use floor instead of subtract for DEM to handle negative elevations properly
    dem_binned = dem.divide(dem_bin_interval).floor().int()
    
    # Add a constant band with value 1 for each pixel (for counting)
    ones = ee.Image.constant(1)
    
    # Create a combined image with both binned values and the constant band
    binned_image = ee.Image.cat([ones, lat_binned, dem_binned]).rename(['count', 'lat_bin', 'dem_bin'])
    
    # Create a reducer for the 2D histogram
    reducer = ee.Reducer.sum().group(
        groupField=1,  # lat_bin is now at index 1
        groupName='lat_bin'
    ).group(
        groupField=2,  # dem_bin is now at index 2
        groupName='dem_bin'
    )
    
    # Calculate the histogram within the continent boundary
    histogram = binned_image.reduceRegion(
        reducer=reducer,
        geometry=continent_ee.geometry(),
        scale=1000,  # Use 1km scale for efficiency
        maxPixels=1e12,
        bestEffort=True
    )
    
    # Convert the complex EE histogram format to a simple 2D array
    hist_array = np.zeros((len(lat_coords), len(dem_coords)))
    
    # Get the histogram result from Earth Engine
    result_dict = histogram.getInfo()
    
    # Add debugging for the first few histogram entries
    if continent_name == continents[0]:
        print(f"Result keys: {list(result_dict.keys())}")
        if 'groups' in result_dict and result_dict['groups']:
            print(f"First group keys: {list(result_dict['groups'][0].keys())}")
            # Print actual lat_bin and dem_bin values for debugging
            if len(result_dict['groups']) > 0 and 'groups' in result_dict['groups'][0]:
                print(f"Sample bin values:")
                for i, dem_group in enumerate(result_dict['groups'][:3]):  # First 3 dem groups
                    if 'dem_bin' in dem_group and 'groups' in dem_group:
                        print(f"  DEM bin: {dem_group['dem_bin']}")
                        for j, lat_group in enumerate(dem_group['groups'][:3]):  # First 3 lat groups
                            if 'lat_bin' in lat_group:
                                real_lat = (int(lat_group['lat_bin']) * lat_bin_interval) + lat_bin_low
                                print(f"    Lat bin: {lat_group['lat_bin']} (corresponds to {real_lat}°)")
                                print(f"    Sum: {lat_group.get('sum', 0)}")
    
    # Parse the nested structure
    if 'groups' in result_dict:
        for dem_group in result_dict['groups']:
            if 'dem_bin' in dem_group:
                dem_idx = int(dem_group['dem_bin'])
                if dem_idx >= len(dem_coords):
                    print(f"Warning: dem_bin {dem_idx} exceeds array bounds")
                    continue
            else:
                continue
                
            for lat_group in dem_group.get('groups', []):
                if 'lat_bin' in lat_group:
                    lat_value = int(lat_group['lat_bin'])
                    
                    # Convert from Earth Engine bin index to actual latitude
                    actual_lat = (lat_value * lat_bin_interval) - 90
                    
                    # Convert from actual latitude to our array index
                    lat_idx = int((actual_lat + 90) / lat_bin_interval)
                    
                    if lat_idx < 0 or lat_idx >= len(lat_coords):
                        print(f"Warning: Latitude index {lat_idx} from bin {lat_value} (actual lat {actual_lat}°) out of bounds")
                        continue
                else:
                    continue
                
                # Check bounds and store the sum
                if 0 <= lat_idx < len(lat_coords) and 0 <= dem_idx < len(dem_coords):
                    hist_array[lat_idx, dem_idx] = lat_group.get('sum', 0)
    
    # Also print the min/max lat_bin values found in the data
    if 'groups' in result_dict:
        lat_bins_found = []
        for dem_group in result_dict['groups']:
            for lat_group in dem_group.get('groups', []):
                if 'lat_bin' in lat_group:
                    lat_bins_found.append(int(lat_group['lat_bin']))
        
        if lat_bins_found:
            min_lat_bin = min(lat_bins_found)
            max_lat_bin = max(lat_bins_found)
            print(f"For {continent_name}, latitude bins range: {min_lat_bin} to {max_lat_bin}")
            print(f"This corresponds to latitudes: {(min_lat_bin * lat_bin_interval) + lat_bin_low}° to {(max_lat_bin * lat_bin_interval) + lat_bin_low}°")
    
    # Rest of your existing function
    return hist_array

# Process each continent
for i, continent in enumerate(continents):
    print(f"Processing {continent}...")
    hist_data[i, :, :] = compute_continent_histogram(continent)



# Create xarray dataset
elev_lat_counts_ds = xr.Dataset(
    data_vars={
        'pixel_count': (('continent', 'latitude', 'elevation'), hist_data)
    },
    coords={
        'continent': continents,
        'latitude': lat_coords,
        'elevation': dem_coords
    }
)

# Add metadata
elev_lat_counts_ds.latitude.attrs['units'] = 'degrees'
elev_lat_counts_ds.elevation.attrs['units'] = 'meters'
elev_lat_counts_ds.pixel_count.attrs['description'] = 'Count of pixels in each latitude-elevation bin'
elev_lat_counts_ds.attrs['description'] = '2D histogram of global land area by continent, latitude and elevation'

elev_lat_counts_ds

In [ ]:
# add elev_lat_counts_ds to merged_ds
# rename pixel_count to dem_pixel_count
merged_with_elev_ds = xr.merge([merged_ds, elev_lat_counts_ds],join='outer') 
merged_with_elev_ds = merged_with_elev_ds.rename({'pixel_count': 'dem_pixel_count'})
merged_with_elev_ds

In [ ]:
merged_with_elev_ds.to_netcdf(f'aggregated_results/continents/fcf_lte_50/{config.version}/all_continents.nc')